In [2]:
import pandas as pd
from feast import FeatureStore
import mlflow
import dagshub

import random
import numpy as np

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score, f1_score


dagshub.init(repo_owner='kerasPro', repo_name='ML2_Clase', mlflow=True)

Initialized MLflow to track repo "kerasPro/ML2_Clase"

Repository kerasPro/ML2_Clase initialized!

In [33]:
fs = FeatureStore("../feast_service/fs_ml2/feature_repo")

In [45]:
entity_df = pd.DataFrame.from_dict(
        {
            # entity's join key -> entity values
            "booking_id": pd.read_parquet("../feast_service/fs_ml2/feature_repo/data/booking_features.parquet")["booking_id"][:1000],
            "kpi1": [ np.random.normal(0) for i in range(1000)],
            "kpi2": [ np.random.normal(0) for i in range(1000)],
        },
    )
entity_df

,booking_id,kpi1,kpi2
0,0017ea8e-8292-4b8c-b67a-6bac03adf2b2,0.671454,1.257507
1,f24c05d7-ac6d-47a6-8879-b284d94e5fb1,-0.886905,0.454240
2,d5664c5c-e8a4-4f3d-93c6-bccb9816c61f,-0.279562,-0.006760
3,564056f2-e41e-455f-a914-4bef83b2d004,0.122951,-0.798522
4,e1260e45-649c-40c5-af11-a94acf06abec,-0.212001,1.224296
...,...,...,...
995,cc9c37f9-8370-499e-83a1-ac002ce52c3d,-0.425583,1.344649
996,5ca41a7e-a9e7-45ff-8e2b-864064ec0fb0,0.477465,-0.452418
997,56d7e030-8298-4b6d-ab38-b5e4e0232074,1.061128,-0.013405
998,4ee36b3c-96f1-4ccd-9a5c-ad9632504ec0,-0.609664,1.500225


## Online para prediccion, pero lo usaremos de ejemplo para ver como transforma todo

In [60]:
feature_service = fs.get_feature_service("feature_service_all_data")

In [61]:
online_features = fs.get_online_features(
    features=feature_service,  # Es más limpio pasar el servicio directamente
    entity_rows=entity_df.to_dict(orient="records")
).to_dict()

In [63]:
df = pd.DataFrame.from_dict(online_features)
df = df.merge(entity_df, on="booking_id")
df["target"] = [random.choice([0,1]) for i in range(1000)]
df

,booking_id,great_feature2,great_feature1,great_feature1_kpi1,great_feature2_kpi2,kpi1,kpi2,target
0,0017ea8e-8292-4b8c-b67a-6bac03adf2b2,-1.214780,-1.073414,-0.720748,-1.527594,0.671454,1.257507,1
1,f24c05d7-ac6d-47a6-8879-b284d94e5fb1,1.666868,0.206829,-0.183438,0.757158,-0.886905,0.454240,0
2,d5664c5c-e8a4-4f3d-93c6-bccb9816c61f,-1.425459,3.617122,-1.011212,0.009636,-0.279562,-0.006760,1
3,564056f2-e41e-455f-a914-4bef83b2d004,1.229981,1.661425,0.204273,-0.982168,0.122951,-0.798522,0
4,e1260e45-649c-40c5-af11-a94acf06abec,-1.139403,-1.230793,0.260930,-1.394967,-0.212001,1.224296,0
...,...,...,...,...,...,...,...,...
995,cc9c37f9-8370-499e-83a1-ac002ce52c3d,-0.841111,-1.065014,0.453252,-1.130999,-0.425583,1.344649,0
996,5ca41a7e-a9e7-45ff-8e2b-864064ec0fb0,1.641057,-0.456712,-0.218064,-0.742444,0.477465,-0.452418,0
997,56d7e030-8298-4b6d-ab38-b5e4e0232074,-0.607280,-0.885785,-0.939931,0.008141,1.061128,-0.013405,0
998,4ee36b3c-96f1-4ccd-9a5c-ad9632504ec0,-0.278335,0.676371,-0.412359,-0.417565,-0.609664,1.500225,0


### Preprocesamiento

In [64]:
X, y = df.drop("target", axis=1), df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=100)

### MLflow

In [65]:
mlflow.set_experiment("ML2 - Feast + MLFLOW  1000")

2025/10/22 20:55:16 INFO mlflow.tracking.fluent: Experiment with name 'ML2 - Feast + MLFLOW  1000' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/310e65fd4e634ac08fbfd3e381762087', creation_time=1761184516655, experiment_id='1', last_update_time=1761184516655, lifecycle_stage='active', name='ML2 - Feast + MLFLOW  1000', tags={}>

In [66]:
mlflow.autolog(log_models=True,)
with mlflow.start_run(run_name="Baseline - Dummy Classifier - Con MAS métricas") as run:

    algorithm = DummyClassifier()
    algorithm.fit(X_train, y_train)

    predictions = algorithm.predict(X_test)

    _accuracy_score = accuracy_score(y_test, predictions)
    _f1_score = f1_score(y_test, predictions)
    
    mlflow.log_metrics(
        {
            "accuracy": _accuracy_score,
            "f1": _f1_score,
            "metrica_dsrp": 100
        }   
    )

2025/10/22 20:56:11 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 0.24.1 <= scikit-learn <= 1.6.1, but the installed version is 1.7.2. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2025/10/22 20:56:12 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
/home/keras/wordspaces/ml2_clases/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2025/10/22 20:56:16 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/home/keras/wordspaces/ml2_clases/.venv/lib/python3.12/site-packages/mlflow/models/model.py:365: DeprecationWarning: datetime.date

🏃 View run Baseline - Dummy Classifier - Con MAS métricas at: https://dagshub.com/kerasPro/ML2_Clase.mlflow/#/experiments/1/runs/78f5bee68ec54f5fb4acd579825b3146
🧪 View experiment at: https://dagshub.com/kerasPro/ML2_Clase.mlflow/#/experiments/1
